# 05 Seasonality Analysis

This notebook analyzes monthly revenue, quarterly patterns, and seasonal discounting with focus on Black Friday and Christmas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

# File paths
orders_path = '/mnt/data/orders_cl.csv'
products_path = '/mnt/data/products_cl(1).csv'
orderlines_path = '/mnt/data/orderlines_qu.csv'
brands_path = '/mnt/data/brands(1).csv'

# Load datasets
orders_cl = pd.read_csv(orders_path)
products_cl = pd.read_csv(products_path)
orderlines_qu = pd.read_csv(orderlines_path)
brands_df = pd.read_csv(brands_path)

print('orders_cl:', orders_cl.shape)
print('products_cl:', products_cl.shape)
print('orderlines_qu:', orderlines_qu.shape)
print('brands_df:', brands_df.shape)

orders_clean = orders_cl.drop_duplicates().copy()
products_clean = products_cl.drop_duplicates(subset='sku').copy()
orderlines_clean = orderlines_qu.drop_duplicates().copy()
brands_clean = brands_df.drop_duplicates().copy()

orders_clean['created_date'] = pd.to_datetime(orders_clean['created_date'], errors='coerce')
orderlines_clean['date'] = pd.to_datetime(orderlines_clean['date'], errors='coerce')
orders_clean['total_paid'] = pd.to_numeric(orders_clean['total_paid'], errors='coerce')
products_clean['price'] = pd.to_numeric(products_clean['price'], errors='coerce')
orderlines_clean['unit_price'] = pd.to_numeric(orderlines_clean['unit_price'], errors='coerce')
orderlines_clean['product_quantity'] = pd.to_numeric(orderlines_clean['product_quantity'], errors='coerce')

orders_clean = orders_clean.dropna(subset=['order_id', 'created_date', 'total_paid', 'state'])
products_clean = products_clean.dropna(subset=['sku', 'name', 'price'])
orderlines_clean = orderlines_clean.dropna(subset=['id_order', 'sku', 'unit_price', 'product_quantity', 'date'])
products_clean = products_clean[products_clean['price'] > 0]
orderlines_clean = orderlines_clean[(orderlines_clean['unit_price'] > 0) & (orderlines_clean['product_quantity'] > 0)]

products_clean['brand_code'] = products_clean['sku'].str[:3]
products_clean = products_clean.merge(brands_clean, left_on='brand_code', right_on='short', how='left')
products_clean = products_clean.rename(columns={'long': 'brand_name'})

sales_df = orderlines_clean.merge(
    products_clean[['sku', 'price', 'brand_name']],
    on='sku',
    how='left'
).merge(
    orders_clean[['order_id', 'created_date', 'state']],
    left_on='id_order',
    right_on='order_id',
    how='left'
)

sales_df['revenue'] = sales_df['unit_price'] * sales_df['product_quantity']
sales_df['discount_pct'] = ((sales_df['price'] - sales_df['unit_price']) / sales_df['price']) * 100
sales_df['year_month'] = sales_df['created_date'].dt.to_period('M').astype(str)
sales_df['quarter'] = sales_df['created_date'].dt.quarter


## 1. Monthly revenue

In [ ]:
monthly_revenue = sales_df.groupby('year_month')['revenue'].sum()

plt.figure(figsize=(10,4))
monthly_revenue.plot()
plt.title('Monthly revenue')
plt.ylabel('Revenue (€)')
plt.xticks(rotation=45)
plt.show()

## 2. Quarterly items sold (completed orders only)

In [ ]:
completed_sales = sales_df[sales_df['state'] == 'Completed'].copy()
quarter_items = completed_sales.groupby('quarter')['product_quantity'].sum()

display(quarter_items)

plt.figure(figsize=(6,4))
quarter_items.plot(kind='bar')
plt.title('Items sold by quarter (Completed orders)')
plt.ylabel('Items sold')
plt.xticks(rotation=0)
plt.show()

## 3. Monthly average discount for Apple products

In [ ]:
apple_sales = sales_df[sales_df['brand_name'].str.contains('Apple', case=False, na=False)].copy()
apple_monthly_discount = apple_sales.groupby('year_month')['discount_pct'].mean()

plt.figure(figsize=(10,4))
apple_monthly_discount.plot()
plt.title('Average monthly discount for Apple products')
plt.ylabel('Average discount (%)')
plt.xticks(rotation=45)
plt.show()

## 4. Black Friday and Christmas focus

In [ ]:
sales_df['month'] = sales_df['created_date'].dt.month
seasonal_summary = sales_df.groupby('month').agg(
    revenue=('revenue', 'sum'),
    avg_discount_pct=('discount_pct', 'mean'),
    items_sold=('product_quantity', 'sum')
).round(2)

display(seasonal_summary)

plt.figure(figsize=(10,4))
seasonal_summary['revenue'].plot(kind='bar')
plt.title('Revenue by calendar month')
plt.ylabel('Revenue (€)')
plt.xticks(rotation=0)
plt.show()

## 5. Business interpretation

Use this markdown cell to connect Q4 behavior to Black Friday and Christmas.